# LangChain — Aulas 01 a 05

Notebook único, uma seção por aula e por vídeo. Localmente as chaves vêm do arquivo `.env` (veja `.example.env`); no Colab, dos Secrets.

In [ ]:
# Instalação (só roda no Google Colab; localmente use o requirements.txt)
import sys, os
if "google.colab" in sys.modules:
    %pip install -q langchain langchain-core langchain-classic langchain-community langchain-text-splitters langchain-google-genai langchain-openai langgraph langgraph-checkpoint-sqlite faiss-cpu tavily-python python-dotenv fastapi httpx pydantic
    # Chaves: cadastre em "Secrets" (ícone de chave) no Colab
    from google.colab import userdata
    for _k in ("OPENAI_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY", "API_KEY_ANALISTA", "API_KEY_DIRETORIA", "OPENAI_CHAT_MODEL", "GEMINI_CHAT_MODEL"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass

---
# Aula 01

## video01

In [ ]:
import os

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# CONFIGURAÇÃO DE AMBIENTE
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente GOOGLE_API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

print("--- Inicializando o Modelo e a Prompt Template ---")

# 2. Definindo o modelo de linguagem (LLM)
model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente técnico especialista em arquitetura de software de Inteligência Artificial. Seja direto e profissional."),
    ("user", "Explique brevemente o conceito de {topico} e por que ele é modular.")
])

output_parser = StrOutputParser()

chain = prompt_template | model | output_parser

topico_exemplo = "LangChain Expression Language (LCEL)"
print(f"\n[Executando Chain] Solicitando explicação sobre: {topico_exemplo}...")  

resposta = chain.invoke({"topico": topico_exemplo})

print("\n=== RESPOSTA DA CHAIN INTERNA ===")
print(resposta)

## video02

In [ ]:
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

# Silencia avisos de schema não suportado para limpar o console
logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)

load_dotenv()

print("--- Configurando um Agente com Memória e Ferramentas Customizadas ---")

@tool
def calcular_custo_tokens(quantidade_tokens: int, modelo: str) -> str:
    """Calcula o custo estimado em dólares com base na quantidade de tokens e no modelo especificado."""
    modelo = modelo.lower()
    if "gpt-4o-mini" in modelo:
        custo = (quantidade_tokens / 1_000_000) * 0.15 # Exemplo de preço hipotético por milhão
        return f"O custo estimado para {quantidade_tokens} tokens no gpt-4o-mini é de ${custo:.6f} USD."
    else:
        return "Modelo não mapeado para cálculo de custo local."

# Lista de ferramentas disponíveis para o agente
tools = [calcular_custo_tokens]

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

# Criando o Prompt do Agente contendo um espaço reservado para o histórico (Memória)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um agente financeiro de IA. Use as ferramentas disponíveis para responder dúvidas sobre custos se necessário."),
    MessagesPlaceholder(variable_name="chat_history"), # Onde o histórico da conversa reside
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # Onde o agente "pensa" e planeja as ações
])

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

historico_conversa = [
    HumanMessage(content="Olá, meu nome é Carlos e estou construindo um assistente."),
    AIMessage(content="Olá Carlos! Como posso te ajudar com os custos ou arquitetura do seu assistente hoje?")
]

pergunta_usuario = "Se meu sistema rodar 5.000.000 de tokens no gpt-4o-mini, quanto vou gastar?"

print(f"\n[Usuário]: {pergunta_usuario}")
resposta_final = agent_executor.invoke({
    "input": pergunta_usuario,
    "chat_history": historico_conversa
})

print("\n=== RESPOSTA FINAL DO AGENTE ===")
print(resposta_final["output"])

## video03

In [ ]:
# Script_Video_03.py
import os
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilyAnswer
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)

load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") and os.getenv("TAVILY_API_KEY")):
    raise ValueError("A variável de ambiente GOOGLE_API_KEY e TAVILY_API_KEY não foram encontradas.\
                      Crie um arquivo .env e adicione-a.")

print("--- Iniciando o Projeto Prático: Answer Engine ---")

# 1. Instanciando uma ferramenta de busca em tempo real na internet
busca_internet = TavilyAnswer()
ferramentas_projeto = [busca_internet]

# 2. Inicializando a LLM que atuará como controladora
llm_controladora = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", 
    temperature=0
)

# 3. Definindo o prompt de comportamento do Answer Engine
prompt_engine = ChatPromptTemplate.from_messages([
    ("system", (
        "Você é o 'Answer Engine', um motor de respostas avançado integrado à internet. "
        "Sempre que o usuário perguntar sobre fatos recentes, ações de mercado, notícias ou cotações, "
        "use a ferramenta de busca para trazer dados verídicos e cite as descobertas de forma resumida."
    )),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 4. Montando o agente focado em Tool Binding (vincular ferramentas ao modelo)
engine_agent = create_tool_calling_agent(llm_controladora, ferramentas_projeto, prompt_engine)

# 5. Criando o executor do Answer Engine
answer_engine_executor = AgentExecutor(
    agent=engine_agent, 
    tools=ferramentas_projeto, 
    verbose=True, # Exibe no terminal os passos de "Thought -> Action -> Observation"
    handle_parsing_errors=True # Garante resiliência caso o formato de saída quebre
)

# 6. Execução do teste prático (Uma pergunta factual recente que a LLM não saberia sem a ferramenta)
pergunta_atual = "Quem ganhou o último campeonato de Fórmula 1 e qual foi a pontuação final?"

print(f"\n[Consultando Answer Engine]: {pergunta_atual}")
resultado_engine = answer_engine_executor.invoke({"input": pergunta_atual})

print("\n=== RESPOSTA ENRICHED (COM BUSCA EM TEMPO REAL) ===")
print(resultado_engine["output"])

---
# Aula 02

## video01

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
import os

# CONFIGURAÇÃO DE AMBIENTE
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# 1. Configuração das credenciais (Mantenha suas chaves seguras)
# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente para a API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)
parser = StrOutputParser()

# 2. Criação dos Componentes da Chain
prompt_gerador = ChatPromptTemplate.from_template(
    "Gere uma breve descrição técnica sobre o conceito de: {topico}"
)

# 3. Função customizada usando RunnableLambda (Sanitização)
prompt_revisor = ChatPromptTemplate.from_template(
    "Melhore o seguinte texto técnico, corrigindo eventuais termos e tornando-o mais formal:\n\n{texto_gerado}"
)

def limpar_texto(texto: str) -> str:
    # Simula uma limpeza ou remoção de caracteres indesejados
    return texto.strip().replace("  ", " ")

runnable_limpeza = RunnableLambda(limpar_texto)

# 4. Construindo a Chain com o Operador Pipe (|)
chain = (
    prompt_gerador
    | model 
    | parser 
    | runnable_limpeza
    | {"texto_gerado": RunnablePassthrough()}  # Mantém o dado vivo no dicionário para o próximo prompt
    | prompt_revisor 
    | model 
    | parser
)

print("=== 1. DEMONSTRANDO .invoke() ===")
resultado_invoke = chain.invoke({"topico": "LangChain Expression Language (LCEL)"})
print(resultado_invoke)
print("\n" + "="*40 + "\n")

print("=== 2. DEMONSTRANDO .stream() ===")
# Excelente para mostrar o surgimento de tokens em tempo real na gravação
for chunk in chain.stream({"topico": "RunnableParallel"}):
    print(chunk, end="", flush=True)
print("\n\n" + "="*40 + "\n")

print("=== 3. DEMONSTRANDO .batch() ===")
# Execução paralela nativa de múltiplos inputs
inputs = [
    {"topico": "Large Language Models"},
    {"topico": "API Idempotency"}
]

resultados_batch = chain.batch(inputs)
for i, res in enumerate(resultados_batch):
    print(f"Resultado {i+1}:\n{res}\n")

## video02

In [ ]:
import os
from pydantic import BaseModel, Field, field_validator
import unicodedata
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv

load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente para a API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

# 1. Configuração do Modelo de Classificação Inteligente (Pydantic)
class RoteadorSuporte(BaseModel):
    """Classifica a intenção do usuário para direcionar ao setor correto.

    Normaliza o valor de `setor` para lowercase antes da validação, assim
    aceitando respostas como "Financeiro" ou "financeiro" vindas do LLM.
    """
    setor: str = Field(
        description="O setor para onde a dúvida deve ser enviada baseado no contexto."
    )

    @field_validator("setor", mode="before")
    def _normalize_setor(cls, v):
        if isinstance(v, str):
            # Remove acentos e normalize para facilitar mapeamento
            s = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")
            s = s.strip().lower()
            # Map common variations to canonical values
            if "finance" in s:
                return "financeiro"
            if "tecn" in s or "suporte" in s or "tecnico" in s:
                return "tecnico"
            return s
        return v

    @field_validator("setor")
    def _validate_setor(cls, v):
        if v not in ("financeiro", "tecnico"):
            raise ValueError("setor must be 'financeiro' or 'tecnico'")
        return v

model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

llm_classificador = model.with_structured_output(RoteadorSuporte)

# 2. Definição das Sub-Chains Especializadas
prompt_financeiro = ChatPromptTemplate.from_template(
    "Você é um especialista financeiro. Responda a seguinte dúvida sobre faturamento de forma clara e cordial: {pergunta}"
)
prompt_tecnico = ChatPromptTemplate.from_template(
    "Você é um engenheiro de software sênior. Responda a seguinte dúvida de TI de forma técnica e objetiva: {pergunta}"
)

parser = StrOutputParser()

chain_financeira = prompt_financeiro | model | parser
chain_tecnica = prompt_tecnico | model | parser

# 3. Lógica do Roteador (Decisor Dinâmico)
def rotear_consulta(dados):
    intencao = dados["intencao"].setor
    pergunta_original = dados["pergunta"]
    
    print(f"\n[Roteador] -> Intenção detectada: {intencao.upper()}")
    
    if intencao == "financeiro":
        return chain_financeira.invoke({"pergunta": pergunta_original})
    elif intencao == "tecnico":
        return chain_tecnica.invoke({"pergunta": pergunta_original})

# 4. Orquestração da Chain de Roteamento Completa
# Usamos RunnableParallel de forma implícita passando o dicionário
chain_principal = {
    "intencao": ChatPromptTemplate.from_template("{pergunta}") | llm_classificador,
    "pergunta": RunnablePassthrough()
} | RunnableLambda(rotear_consulta)

# print("=== TESTE 1: INPUT FINANCEIRO ===")
# pergunta_1 = "Minha fatura veio duplicada este mês, como solicito o estorno?"
# resposta_1 = chain_principal.invoke({"pergunta": pergunta_1})
# print(f"Resposta Final:\n{resposta_1}")
# print("\n" + "="*40 + "\n")

print("=== TESTE 2: INPUT TÉCNICO ===")
pergunta_2 = "Estou recebendo erro 500 ao tentar conectar no banco PostgreSQL local."
resposta_2 = chain_principal.invoke({"pergunta": pergunta_2})
print(f"Resposta Final:\n{resposta_2}")

## video03

In [ ]:
import os
import logging
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.globals import set_debug
from langchain_core.callbacks import BaseCallbackHandler

from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)
set_debug(False)

# Checagem básica de credenciais (não interrompe, apenas alerta)
if not os.getenv("OPENAI_API_KEY"):
    logging.warning("OPENAI_API_KEY não encontrada. Chamadas ao OpenAI vão falhar se não configurada.")
if not os.getenv("GOOGLE_API_KEY"):
    logging.warning("GOOGLE_API_KEY não encontrada. Chamadas ao Google Generative AI vão falhar se não configurada.")


# 1. Definição de uma Custom Tool com Tipagem e Docstring Ricas
@tool
def calcular_faturamento_anual(empresa: str, receita_mensal: float) -> str:
    """Calcula a projeção de faturamento anual de uma empresa baseada na receita mensal."""
    # A docstring funciona como a descrição lida pelo agente para decidir o uso da ferramenta
    faturamento_projetado = receita_mensal * 12
    return f"A projeção de faturamento anual para a empresa {empresa} é de R$ {faturamento_projetado:,.2f}"

class LogFallbackHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        nome_modelo = serialized.get("id", ["LLM"])[-1]
        print(f"\n🟡 [TENTATIVA] Chamando modelo: {nome_modelo}")

    def on_llm_error(self, error, **kwargs):
        print(f"🔴 [FALHA] Modelo falhou com o erro: {error}")
        print("🔄 [FALLBACK] Redirecionando para o modelo secundário...")

    def on_llm_end(self, response, **kwargs):
        print("🟢 [SUCESSO] LLM respondeu com sucesso!")

tools = [calcular_faturamento_anual]

# 2. Prompt Padrão
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente financeiro corporativo de elite. Utilize as ferramentas fornecidas sempre que precisar de dados calculados."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 3. Modelos
llm_primario = ChatOpenAI(model="gpt-5.4-mini", temperature=0, timeout=0.001, max_retries=0)
llm_secundario = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 4. Criação dos Agentes Individuais com Tool Calling Universal
agent_primario = create_tool_calling_agent(llm_primario, tools, prompt)
agent_secundario = create_tool_calling_agent(llm_secundario, tools, prompt)

# 5. Aplicação do Fallback no Nível do Agente (Runnable)
agent_resiliente = agent_primario.with_fallbacks([agent_secundario])

# 6. Runtime Executor
agent_executor = AgentExecutor(
    agent=agent_resiliente,
    tools=tools,
    verbose=True
)

print("=== EXECUTANDO O AGENTE AUTÔNOMO ===")
consulta = "A empresa AlfaTech fatura 25000 reais por mês. Qual a projeção anual deles?"

try:
    resposta = agent_executor.invoke({"input": consulta}, config={"callbacks": [LogFallbackHandler()]})
    print("\n=== RESPOSTA FINAL DO AGENTE ===")
    # Normaliza formatos de saída comuns
    if isinstance(resposta, dict):
        out = resposta.get("output") or resposta.get("text") or resposta.get("result") or str(resposta)
    else:
        out = str(resposta)
    print(out)
except Exception as e:
    logging.exception("Falha na invocação do agente")
    print("Erro ao executar o agente:", e)

---
# Aula 03

## video01

In [ ]:
import os
import re

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Tenta usar embeddings do Google (Gemini); usa OpenAI como fallback se não disponível
try:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    _HAS_GOOGLE_EMB = True
except Exception:
    _HAS_GOOGLE_EMB = False

try:
    from langchain_openai import OpenAIEmbeddings
    _HAS_OPENAI_EMB = True
except Exception:
    _HAS_OPENAI_EMB = False

MODELO_GEMINI_PADRAO = "gemini-embedding-001"

def _mascarar_segredos(texto: str) -> str:
    """Remove chaves de API que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))

def _nomes_candidatos(modelo: str):
    """Gera as variações aceitas do nome do modelo.
 
    langchain-google-genai >= 3.x aceita o nome puro ('gemini-embedding-001');
    versões 2.x mandam a string crua para a API, que exige o prefixo 'models/'
    e devolve 'unexpected model name format' sem ele. Tentar as duas cobre
    ambos os casos sem depender da versão instalada.
    """
    if modelo.startswith("models/"):
        return [modelo, modelo.removeprefix("models/")]
    return [modelo, f"models/{modelo}"]

def _criar_embeddings_google():
    """Instancia e valida os embeddings do Gemini. Retorna None se não der para usar."""
    if not _HAS_GOOGLE_EMB:
        print(" Pacote langchain-google-genai não instalado.")
        return None
    if not os.environ.get("GOOGLE_API_KEY"):
        print(" GOOGLE_API_KEY não encontrada no ambiente (.env).")
        return None
 
    modelo = os.environ.get("GOOGLE_EMBEDDING_MODEL", MODELO_GEMINI_PADRAO)
 
    for nome in _nomes_candidatos(modelo):
        try:
            embeddings = GoogleGenerativeAIEmbeddings(model=nome)
            # Chamada mínima só para validar credencial + nome do modelo antes de indexar.
            embeddings.embed_query("ping")
            print(f" Usando GoogleGenerativeAIEmbeddings (modelo: {nome})")
            return embeddings
        except Exception as e:
            print(f" Modelo '{nome}' indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")
 
    return None

def _criar_embeddings_openai():
    """Fallback: embeddings da OpenAI. Retorna None se não der para usar."""
    if not _HAS_OPENAI_EMB or not os.environ.get("OPENAI_API_KEY"):
        return None
    try:
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        print(" Usando OpenAIEmbeddings como fallback")
        return embeddings
    except Exception as e:
        print(" Falha ao inicializar OpenAIEmbeddings:", _mascarar_segredos(e))
        return None

def criar_embeddings():
    """Escolhe o provedor de embeddings: Gemini primeiro, OpenAI como reserva."""
    return _criar_embeddings_google() or _criar_embeddings_openai()

def run_aula_1():
    print("--- INICIANDO AULA 3: PIPELINE DE INGESTÃO LOCAL ---")

    # 1. Dados corporativos simulados (Relatório Financeiro)
    documento_corporativo = """
    Relatório de Desempenho Financeiro - Indústrias ACME (Maio 2026).
    O faturamento consolidado no último mês atingiu R$ 12.5 milhões, representando uma alta de 8% em relação a abril.
    O principal motor de crescimento foi a divisão de Microcontroladores, que expandiu sua operação global em 15%.
    As despesas operacionais (OPEX) totalizaram R$ 4.2 milhões, mantendo-se dentro do orçamento previsto.
    A projeção para o próximo trimestre é manter a taxa de crescimento acima de dois dígitos.
    """

    # 2. Chunking (Fragmentação Inteligente) com Overlap para não perder contexto
    print("\n[Passo 1] Fragmentando o texto...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=30
    )
    chunks = text_splitter.split_text(documento_corporativo)
    for i, chunk in enumerate(chunks):
        print(f" Chunk {i+1}: {chunk.strip()}")

    # 3. Inicialização do modelo de Embedding
    print("\n[Passo 2] Gerando vetores (Gemini/Google ou OpenAI, conforme disponível)...")
    embeddings = criar_embeddings()
    if embeddings is None:
        print("ERRO: Nenhum provedor de embeddings disponível. "
              "Configure GOOGLE_API_KEY (ou OPENAI_API_KEY) e as bibliotecas necessárias.")
        return

    # 4. Criação do Banco Vetorial local com FAISS
    print("\n[Passo 3] Indexando vetores no FAISS...")
    try:
        vector_store = FAISS.from_texts(chunks, embeddings)
    except Exception as e:
        print("ERRO ao indexar os vetores:", _mascarar_segredos(e))
        return

    # 5. Teste de Busca Semântica (Simulando a pergunta do usuário)
    query = "Quanto a empresa faturou no mês passado?"
    print(f"\n[Passo 4] Executando busca semântica para: '{query}'")

    resultados = vector_store.similarity_search(query, k=1)

    print("\n[Resultado da Busca] Documento mais relevante encontrado:")
    print(f" -> {resultados[0].page_content.strip()}")

### Execução

In [ ]:
load_dotenv()
run_aula_1()

## video02

In [ ]:
import os
import re

from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

MODELO_EMBEDDING_PADRAO = "gemini-embedding-001"
MODELO_CHAT_PADRAO = "gemini-3.5-flash"
MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")

QTD_VARIACOES = 2
DOCS_POR_CONSULTA = 2

def _mascarar_segredos(texto: str) -> str:
    """Remove chaves de API que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))


def _nomes_candidatos(modelo: str):
    """Nome puro e com prefixo 'models/'.

    langchain-google-genai >= 3.x normaliza sozinho; a 2.x manda a string crua
    para a API, que exige o prefixo e devolve 'unexpected model name format'.
    """
    if modelo.startswith("models/"):
        return [modelo, modelo.removeprefix("models/")]
    return [modelo, f"models/{modelo}"]

def criar_embeddings():
    """Instancia e valida os embeddings do Gemini antes de indexar qualquer coisa."""
    modelo = os.environ.get("GOOGLE_EMBEDDING_MODEL", MODELO_EMBEDDING_PADRAO)

    for nome in _nomes_candidatos(modelo):
        try:
            embeddings = GoogleGenerativeAIEmbeddings(model=nome)
            embeddings.embed_query("ping")  # valida credencial + nome do modelo
            print(f" Embeddings OK (modelo: {nome})")
            return embeddings
        except Exception as e:
            print(f" Modelo '{nome}' indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")

    return None

def _extrair_variacoes(texto: str, limite: int):
    """Transforma a resposta da LLM em uma lista de perguntas, sem numeração nem linhas vazias."""
    variacoes = []
    for linha in texto.splitlines():
        limpa = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s*", "", linha).strip()
        if limpa:
            variacoes.append(limpa)
    return variacoes[:limite]

def buscar_multi_query(vector_store, consultas, k: int):
    """Busca cada consulta e devolve a união dos documentos, sem repetidos.

    O dict preserva a ordem de inserção, então os documentos da pergunta
    original aparecem primeiro no contexto.
    """
    unicos = {}
    for consulta in consultas:
        for doc in vector_store.similarity_search(consulta, k=k):
            unicos.setdefault(doc.page_content, doc)
    return list(unicos.values())

def run_aula_2():
    print("--- INICIANDO AULA 2: ADVANCED RAG (MULTI-QUERY) ---")

    embeddings = criar_embeddings()
    if embeddings is None:
        print("ERRO: não foi possível inicializar os embeddings. Verifique GOOGLE_API_KEY no .env.")
        return

    llm = ChatGoogleGenerativeAI(model=os.environ.get("GOOGLE_CHAT_MODEL", MODELO_CHAT_PADRAO))

    # 1. Base de conhecimento local pré-indexada
    textos = [
        "O faturamento consolidado da ACME em Maio de 2026 foi de R$ 12.5 milhões.",
        "A divisão de Microcontroladores teve expansão de 15% na operação global.",
        "O Diretor Executivo da ACME em 2026 é o Engenheiro Carlos Eduardo."
    ]

    try:
        vector_store = FAISS.from_texts(textos, embeddings)
    except Exception as e:
        print("ERRO ao indexar os vetores:", _mascarar_segredos(e))
        return

    # 2. Query Transformation: a LLM gera variações para aumentar o Recall
    pergunta_usuario = "Quem lidera a empresa ACME e como foi o lucro?"
    print(f"\n[Pergunta Original]: {pergunta_usuario}")

    prompt_multi_query = PromptTemplate.from_template(
        "Você é um assistente técnico de IA. Gere {qtd} variações alternativas da seguinte pergunta "
        "para ajudar na busca em um banco de dados vetorial.\n"
        "Responda APENAS com as perguntas, uma por linha, sem numeração e sem comentários.\n"
        "Pergunta original: {pergunta}"
    )

    chain_variacoes = prompt_multi_query | llm | StrOutputParser()

    try:
        resposta_variacoes = chain_variacoes.invoke(
            {"pergunta": pergunta_usuario, "qtd": QTD_VARIACOES}
        )
    except Exception as e:
        print("ERRO ao gerar variações:", _mascarar_segredos(e))
        return

    variacoes = _extrair_variacoes(resposta_variacoes, QTD_VARIACOES)

    print("\n[Query Transformation - Variações Geradas pela LLM]:")
    for v in variacoes:
        print(f" - {v}")

    # 3. Retrieval: busca a pergunta original E cada variação, unindo os resultados
    print("\n[Retrieval] Buscando informações relevantes...")
    consultas = [pergunta_usuario] + variacoes
    documentos_recuperados = buscar_multi_query(vector_store, consultas, k=DOCS_POR_CONSULTA)

    contexto = "\n".join(doc.page_content for doc in documentos_recuperados)
    print(f"{len(documentos_recuperados)} documento(s) único(s) recuperado(s):\n{contexto}")

    # 4. Generation: resposta fundamentada apenas no contexto recuperado
    prompt_rag = PromptTemplate.from_template(
        "Você é um assistente corporativo. Responda à pergunta do usuário utilizando APENAS o contexto fornecido abaixo.\n"
        "Se não souber a resposta com base no contexto, diga honestamente que não encontrou a informação.\n\n"
        "Contexto:\n{contexto}\n\n"
        "Pergunta: {pergunta}\n\n"
        "Resposta Resumida e Direta:"
    )

    chain_rag = prompt_rag | llm | StrOutputParser()

    try:
        resposta_final = chain_rag.invoke({"contexto": contexto, "pergunta": pergunta_usuario})
    except Exception as e:
        print("ERRO ao gerar a resposta final:", _mascarar_segredos(e))
        return

    print("\n[Resposta Final Fundamentada da LLM]:")
    print(resposta_final)

### Execução

In [ ]:
load_dotenv()
run_aula_2()

## video03

In [ ]:
import os
from pathlib import Path
import re
import sqlite3
from contextlib import closing

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory

# Fallback opcional: só é usado se o pacote estiver instalado.
try:
    from langchain_openai import ChatOpenAI
    _HAS_OPENAI = True
except Exception:
    _HAS_OPENAI = False

MODELO_CHAT_PADRAO = os.environ.get("GEMINI_CHAT_MODEL", "")
MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")


# Arquivo único no diretório do projeto. Dá para abrir no VS Code e ver as linhas.
DATABASE = os.environ.get("SQLITE_PATH", "memoria_chat.db")
CAMINHO_BANCO = Path(Path(__file__).parent, DATABASE)
CAMINHO_BANCO.touch(exist_ok=True)

TABELA_HISTORICO = "historico_conversas"

TIPOS_PERSISTIDOS = {"human": HumanMessage, "ai": AIMessage}


def _mascarar_segredos(texto: str) -> str:
    """Remove credenciais que possam aparecer em mensagens de erro antes de imprimir."""
    return re.sub(r"(key=|AIza|sk-)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))


def _validar(llm, rotulo: str):
    """Faz uma chamada mínima para confirmar que o provedor responde.

    É aqui que a cota estourada aparece (429), em vez de estourar no meio da
    conversa depois de já ter gravado o turno do usuário no banco.
    """
    try:
        llm.invoke("ping")
        print(f" LLM OK: {rotulo}")
        return llm
    except Exception as e:
        print(f" {rotulo} indisponível -> {type(e).__name__}: {_mascarar_segredos(e)}")
        return None

def _criar_llm_google():
    if not os.environ.get("GOOGLE_API_KEY"):
        print(" GOOGLE_API_KEY não encontrada no .env.")
        return None
    modelo = os.environ.get("GOOGLE_CHAT_MODEL", MODELO_CHAT_PADRAO)
    return _validar(ChatGoogleGenerativeAI(model=modelo), f"Google / {modelo}")


def _criar_llm_openai():
    if not _HAS_OPENAI:
        print(" Pacote langchain-openai não instalado.")
        return None
    if not os.environ.get("OPENAI_API_KEY"):
        print(" OPENAI_API_KEY não encontrada no .env.")
        return None
    modelo = os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO)
    return _validar(ChatOpenAI(model=modelo), f"OpenAI / {modelo}")


def criar_llm():
    """Escolhe o provedor do chat.

    LLM_PROVIDER aceita 'google', 'openai' ou 'auto' (padrão): tenta o Google e
    cai para a OpenAI se ele não responder — cota estourada, chave ausente ou
    modelo indisponível.
    """
    provedor = os.environ.get("LLM_PROVIDER", "auto").lower()

    if provedor == "google":
        return _criar_llm_google()
    if provedor == "openai":
        return _criar_llm_openai()

    return _criar_llm_google() or _criar_llm_openai()

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """Persistência de mensagens em SQLite.

    Abre uma conexão por operação e fecha em seguida. Uma conexão sqlite3 não
    pode ser usada em uma thread diferente da que a criou, e o LangChain executa
    a chain em uma thread pool — por isso nada de conexão guardada em atributo.
    Abrir conexão no SQLite é só abrir um arquivo local, então o custo é
    irrelevante (diferente do MySQL, onde a conexão é um socket de rede).
    """

    def __init__(self, session_id: str, caminho: str = None):
        self.session_id = session_id
        self.caminho = caminho or CAMINHO_BANCO
        self._garantir_tabela()

    def _conectar(self) -> sqlite3.Connection:
        conexao = sqlite3.connect(self.caminho, timeout=5.0)
        conexao.execute("PRAGMA journal_mode=WAL")  # evita 'database is locked'
        return conexao

    def _garantir_tabela(self) -> None:
        with closing(self._conectar()) as conexao, conexao:
            conexao.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {TABELA_HISTORICO} (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL CHECK (role IN ('human', 'ai')),
                    content TEXT NOT NULL,
                    criado_em TEXT DEFAULT CURRENT_TIMESTAMP
                )
                """
            )
            conexao.execute(
                f"CREATE INDEX IF NOT EXISTS idx_sessao ON {TABELA_HISTORICO} (session_id, id)"
            )

    @property
    def messages(self) -> list[BaseMessage]:
        with closing(self._conectar()) as conexao:
            linhas = conexao.execute(
                f"SELECT role, content FROM {TABELA_HISTORICO} "
                "WHERE session_id = ? ORDER BY id ASC",
                (self.session_id,),
            ).fetchall()

        return [
            TIPOS_PERSISTIDOS[role](content=content)
            for role, content in linhas
            if role in TIPOS_PERSISTIDOS
        ] 

    
    def add_messages(self, messages: list[BaseMessage]) -> None:
        linhas = [
            (self.session_id, m.type, m.content)
            for m in messages
            if m.type in TIPOS_PERSISTIDOS
        ]
        if not linhas:
            return

        # 'with conexao' abre transação: commit no sucesso, rollback na exceção.
        with closing(self._conectar()) as conexao, conexao:
            conexao.executemany(
                f"INSERT INTO {TABELA_HISTORICO} (session_id, role, content) VALUES (?, ?, ?)",
                linhas,
            )

    
    def clear(self) -> None:
        with closing(self._conectar()) as conexao, conexao:
            conexao.execute(
                f"DELETE FROM {TABELA_HISTORICO} WHERE session_id = ?",
                (self.session_id,),
            )

def run_aula_3_sqlite():
    print("--- INICIANDO AULA 3: MEMÓRIA PERSISTENTE COM SQLITE ---")
    print(f"Banco: {CAMINHO_BANCO}")

    llm = criar_llm()
    if llm is None:
        print("ERRO: nenhum provedor de LLM disponível. Verifique as chaves no .env.")
        return

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Você é um assistente técnico de banco de dados. Seja breve e amigável."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

    chain = prompt | llm

    chain_com_memoria = RunnableWithMessageHistory(
        chain,
        get_session_history=SQLiteChatMessageHistory,
        input_messages_key="input",
        history_messages_key="history"
    )

    configuracao_usuario = {"configurable": {"session_id": "sessao_aluno_pos_tech"}}

    # ---- TURNO 1 ----
    print("\n[Turno 1] Enviando informações iniciais...")
    input_1 = "Olá professor! Meu nome é Lucas e estou desenvolvendo o projeto final em Python."
    print(f"Aluno: {input_1}")
    resposta_1 = chain_com_memoria.invoke({"input": input_1}, config=configuracao_usuario)
    print(f"Assistente: {resposta_1.content}")

    # ---- TURNO 2 (teste de retenção via banco) ----
    print("\n[Turno 2] Perguntando com base no histórico gravado no SQLite...")
    input_2 = "Com base no que eu te disse, qual é o meu nome e qual linguagem estou usando?"
    print(f"Aluno: {input_2}")
    resposta_2 = chain_com_memoria.invoke({"input": input_2}, config=configuracao_usuario)
    print(f"Assistente: {resposta_2.content}")

    print(f"\n--- Concluído. Abra o arquivo {CAMINHO_BANCO} no VS Code para ver as linhas gravadas. ---")

### Execução

In [ ]:
load_dotenv()

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("OPENAI_API_KEY")):
    print("ERRO: configure GOOGLE_API_KEY ou OPENAI_API_KEY no .env antes de rodar o script.")
else:
    try:
        run_aula_3_sqlite()
    except Exception as e:
        print(f"\nERRO na execução: {type(e).__name__}: {_mascarar_segredos(e)}")

---
# Aula 04

## video01

In [ ]:
"""
1. O schema não é documentação — ele REJEITA entrada inválida em tempo de execução.
2. O schema também é o contrato que o LLM enxerga: restrição no schema reduz
   alucinação, porque o modelo vê os valores permitidos antes de chamar.
3. Os argumentos vêm do LLM, que os inventou a partir de texto do usuário.
   Isso é entrada não confiável chegando no seu banco.
"""

import json
from typing import Literal

from langchain_core.tools import tool
from pydantic import BaseModel, Field

MESES = Literal[
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
]

FILIAIS = Literal["Geral", "São Paulo", "Recife", "Curitiba"]

class ConsultarVendasSchema(BaseModel):
    """Esquema de entrada com validação rigorosa."""

    mes: MESES = Field(..., description="Mês de referência")
    ano: int = Field(..., ge=2020, le=2030, description="Ano com 4 dígitos")
    filial: FILIAIS = Field("Geral", description="Nome da filial para filtrar")

@tool("consultar_vendas_db", args_schema=ConsultarVendasSchema)
def consultar_vendas(mes: str, ano: int, filial: str = "Geral"):
    """Consulta o total de vendas de uma filial em um mês específico.

    Use quando o usuário pedir faturamento, vendas ou performance de um período.
    Não use para projeções futuras nem para despesas.
    """
    # Em produção a query entra aqui — SEMPRE parametrizada:
    #   cursor.execute(
    #       "SELECT SUM(valor) FROM vendas WHERE mes = %s AND ano = %s AND filial = %s",
    #       (mes, ano, filial),
    #   )
    # E a autorização é checada AQUI, comparando com o usuário da sessão —
    # nunca confiando no que o LLM decidiu passar como filial.
    return f"Resultado da consulta para {filial} em {mes}/{ano}: R$ 1.250.000,00"

def demonstrar():
    print("=== O que o LLM enxerga (JSON Schema gerado) ===")
    print(json.dumps(consultar_vendas.args, ensure_ascii=False, indent=2))

    print("\n=== Chamada válida ===")
    print(consultar_vendas.invoke({"mes": "Março", "ano": 2026}))

    print("\n=== Chamadas que o schema rejeita ===")
    invalidas = [
        {"mes": "banana", "ano": 2026},                              # mês inexistente
        {"mes": "Março", "ano": "abcd"},                             # ano não numérico
        {"mes": "Março", "ano": 1800},                               # fora da faixa
        {"mes": "Março", "ano": 2026, "filial": "'; DROP TABLE--"},  # fora da allowlist
    ]
    for argumentos in invalidas:
        try:
            print("ACEITOU (não deveria):", consultar_vendas.invoke(argumentos))
        except Exception as e:
            print(f"rejeitado -> {argumentos}")
            print(f"   {type(e).__name__}")

### Execução

In [ ]:
demonstrar()

## video02

In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")

INSTRUCAO_SISTEMA = (
    "Você é um assistente financeiro especialista. "
    "Use as ferramentas disponíveis para responder. "
    "Se a informação não vier de uma ferramenta, diga que não sabe."
)

# --- Ferramenta do vídeo 01 -------------------------------------------------

MESES = Literal[
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
]

FILIAIS = Literal["Geral", "São Paulo", "Recife", "Curitiba"]


class ConsultarVendasSchema(BaseModel):
    """Esquema de entrada com validação rigorosa."""

    mes: MESES = Field(..., description="Mês de referência")
    ano: int = Field(..., ge=2020, le=2030, description="Ano com 4 dígitos")
    filial: FILIAIS = Field("Geral", description="Nome da filial para filtrar")

@tool("consultar_vendas_db", args_schema=ConsultarVendasSchema)
def consultar_vendas(mes: str, ano: int, filial: str = "Geral"):
    """Consulta o total de vendas de uma filial em um mês específico.

    Use quando o usuário pedir faturamento, vendas ou performance de um período.
    Não use para projeções futuras nem para despesas.
    """
    # Em produção a query entra aqui — SEMPRE parametrizada, e a autorização
    # é checada AQUI contra o usuário da sessão, nunca confiando na filial
    # que o LLM decidiu passar.
    return f"Resultado da consulta para {filial} em {mes}/{ano}: R$ 1.250.000,00"

def criar_agente(llm):
    """O system prompt e o scratchpad são responsabilidade do create_agent."""
    return create_agent(
        llm,
        tools=[consultar_vendas],
        system_prompt=INSTRUCAO_SISTEMA,
    )

def executar_e_mostrar(agente, pergunta: str):
    """Imprime cada passo do loop — é o equivalente ao antigo verbose=True."""
    print(f"\n[Pergunta] {pergunta}\n")

    vistas = 0
    for estado in agente.stream({"messages": [HumanMessage(content=pergunta)]},
                                stream_mode="values"):
        mensagens = estado["messages"]
        for msg in mensagens[vistas:]:
            if msg.type == "ai" and msg.tool_calls:
                for chamada in msg.tool_calls:
                    print(f"  [decisão] chamar {chamada['name']} com {chamada['args']}")
            elif msg.type == "tool":
                print(f"  [ferramenta] {msg.content}")
            elif msg.type == "ai":
                print(f"\n[Resposta] {msg.content}")
        vistas = len(mensagens)

def run_aula_4_video_2():
    print("--- AULA 4 / VÍDEO 02: AGENTE COM TOOL CALLING ---")

    llm = ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))
    agente = criar_agente(llm)

    # Caso 1: a pergunta cabe na ferramenta.
    executar_e_mostrar(agente, "Quanto vendemos na filial São Paulo em Janeiro de 2026?")

    # Caso 2: filial fora da allowlist do vídeo 01. O Pydantic barra, o erro
    # volta ao agente como resultado da ferramenta e ele se corrige.
    executar_e_mostrar(agente, "E quanto vendemos na filial de Manaus em Janeiro de 2026?")

### Execução

In [ ]:
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_4_video_2()

## video03

In [ ]:
import logging
import os
from contextvars import ContextVar
from typing import Literal

from dotenv import load_dotenv
from fastapi import Depends, FastAPI, HTTPException, Security
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import APIKeyHeader
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

logger = logging.getLogger('api-financeira')

DEBUG_ERROS = os.environ.get("DEBUG_ERROS", "0").lower() in {"1", "true", "sim"}

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL", "")

INSTRUCAO_SISTEMA = (
    "Você é um assistente financeiro especialista. "
    "Use as ferramentas disponíveis para responder. "
    "Se a informação não vier de uma ferramenta, diga que não sabe."
)

ORIGENS_PERMITIDAS = os.environ.get("CORS_ORIGENS", "http://localhost:3000").split(",")

# --- Identidade do chamador -------------------------------------------------
# O agente roda dentro do request; a ferramenta precisa saber QUEM perguntou.
# Um ContextVar carrega isso até lá sem passar pelo prompt (prompt não é
# controle de acesso — o LLM pode ignorar qualquer instrução de permissão).

usuario_atual: ContextVar[str] = ContextVar("usuario_atual")

esquema_api_key = APIKeyHeader(name="x-api-key", auto_error=False)

# Em produção isto vem do banco/IdP. Aqui é só para a aula.
FILIAIS_POR_USUARIO = {
    "analista_sp": {"Geral", "São Paulo"},
    "diretoria": {"Geral", "São Paulo", "Recife", "Curitiba"},
}

def autenticar(x_api_key: str = Security(esquema_api_key)) -> str:
    """Troca a chave do header por uma identidade. Deny-by-default."""
    usuario = {
        os.environ.get("API_KEY_ANALISTA", ""): "analista_sp",
        os.environ.get("API_KEY_DIRETORIA", ""): "diretoria",
    }.get(x_api_key)

    if not x_api_key or not usuario:
        raise HTTPException(status_code=401, detail="Credencial inválida.")
    return usuario

# --- Ferramenta -------------------------------------------------------------

MESES = Literal[
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
]

FILIAIS = Literal["Geral", "São Paulo", "Recife", "Curitiba"]


class ConsultarVendasSchema(BaseModel):
    mes: MESES = Field(..., description="Mês de referência")
    ano: int = Field(..., ge=2020, le=2030, description="Ano com 4 dígitos")
    filial: FILIAIS = Field("Geral", description="Nome da filial para filtrar")


@tool("consultar_vendas_db", args_schema=ConsultarVendasSchema)
def consultar_vendas(mes: str, ano: int, filial: str = "Geral"):
    """Consulta o total de vendas de uma filial em um mês específico.

    Use quando o usuário pedir faturamento, vendas ou performance de um período.
    """
    # A autorização é checada AQUI, contra o usuário do request — não contra o
    # que o LLM decidiu pedir. Devolver texto em vez de erro deixa o agente
    # explicar a recusa ao usuário.
    permitidas = FILIAIS_POR_USUARIO.get(usuario_atual.get(""), set())
    if filial not in permitidas:
        return f"Acesso negado: este usuário não pode consultar a filial {filial}."

    # Em produção, query parametrizada:
    #   cursor.execute("SELECT SUM(valor) FROM vendas WHERE mes=%s AND ano=%s AND filial=%s",
    #                  (mes, ano, filial))
    return f"Resultado da consulta para {filial} em {mes}/{ano}: R$ 1.250.000,00"

# --- Agente -----------------------------------------------------------------

def criar_agente(llm=None):
    llm = llm or ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))
    return create_agent(llm, tools=[consultar_vendas], system_prompt=INSTRUCAO_SISTEMA)

# --- Contrato da API --------------------------------------------------------

class PerguntaRequest(BaseModel):
    pergunta: str = Field(..., min_length=3, max_length=500)


class RespostaAnalise(BaseModel):
    resposta: str
    ferramentas_usadas: list[str]
    # Só preenchido com DEBUG_ERROS=1. Mostra COM QUE ARGUMENTOS o LLM decidiu
    # chamar cada ferramenta — é o que revela se ele normalizou a entrada.
    detalhe_chamadas: list[dict] | None = None


app = FastAPI(title="API Financeira Agêntica", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=ORIGENS_PERMITIDAS,
    allow_methods=["POST"],
    allow_headers=["x-api-key", "content-type"],
)

agente = criar_agente()

@app.post("/analise-financeira", response_model=RespostaAnalise)
async def analise_financeira(
    corpo: PerguntaRequest,
    usuario: str = Depends(autenticar),
) -> RespostaAnalise:
    token = usuario_atual.set(usuario)
    try:
        estado = await agente.ainvoke(
            {"messages": [HumanMessage(content=corpo.pergunta)]}
        )
    except Exception as e:
        # O traceback completo vai para o console do uvicorn.
        logger.exception("Falha ao invocar o agente (usuário=%s)", usuario)
        detalhe = (f"{type(e).__name__}: {e}" if DEBUG_ERROS
                   else "Falha ao consultar o assistente.")
        raise HTTPException(status_code=502, detail=detalhe)
    finally:
        usuario_atual.reset(token)
 
    mensagens = estado["messages"]
    chamadas = [
        chamada
        for msg in mensagens
        if msg.type == "ai" and msg.tool_calls
        for chamada in msg.tool_calls
    ]
    return RespostaAnalise(
        resposta=mensagens[-1].content,
        ferramentas_usadas=[c["name"] for c in chamadas],
        detalhe_chamadas=(
            [{"ferramenta": c["name"], "argumentos": c["args"]} for c in chamadas]
            if DEBUG_ERROS else None
        ),
    )

@app.get("/health")
async def health():
    return {"status": "ok"}

### Testando a API no notebook

In [ ]:
# Em vez de subir o servidor (uvicorn), testamos os endpoints direto no notebook.
from fastapi.testclient import TestClient

cliente = TestClient(app)
chave = os.environ.get("API_KEY_ANALISTA", "")

print(cliente.get("/health").json())

# Sem credencial -> 401
print(cliente.post("/analise-financeira", json={"pergunta": "Quanto vendemos em Janeiro de 2026?"}).status_code)

# Com credencial
r = cliente.post("/analise-financeira", headers={"x-api-key": chave},
                 json={"pergunta": "Quanto vendemos em Janeiro de 2026?"})
print(r.status_code, r.json())

---
# Aula 05

## video01

In [ ]:
import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

CATEGORIAS = ("financeiro", "tecnico", "comercial")

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

# ===========================================================================
# 1. O jeito linear (LCEL) — o que as aulas anteriores usaram
# ===========================================================================

def executar_lcel(llm, pergunta: str) -> str:
    """Input -> LLM -> Parse -> Output. Uma via só, sem estado."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Você é um atendente corporativo. Responda em uma frase."),
        ("human", "{pergunta}"),
    ])
    return (prompt | llm | StrOutputParser()).invoke({"pergunta": pergunta})

# ===========================================================================
# 2. O mesmo problema como máquina de estados
# ===========================================================================

class EstadoAtendimento(TypedDict):
    """O contrato de dados do grafo.

    Repare que ele carrega muito mais que 'mensagens': categoria é uma decisão
    intermediária, e trilha é a auditoria de tudo que aconteceu. Num caso real
    entrariam aqui DataFrames, flags de compliance, id do cliente etc.
    """

    pergunta: str
    categoria: str
    resposta: str
    # Annotated + operator.add = REDUCER: o LangGraph SOMA as listas devolvidas
    # por cada nó em vez de sobrescrever. É como o histórico se acumula.
    trilha: Annotated[list[str], operator.add]

def criar_app(llm):
    def classificar(estado: EstadoAtendimento) -> dict:
        """Nó 1: decide a categoria. Devolve SÓ o que mudou."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Classifique a pergunta em exatamente uma palavra, entre: "
             f"{', '.join(CATEGORIAS)}. Responda só a palavra."),
            ("human", "{pergunta}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke(
            {"pergunta": estado["pergunta"]}
        ).strip().lower()

        # O LLM é probabilístico: valide a saída antes de confiar nela.
        categoria = bruto if bruto in CATEGORIAS else "comercial"

        return {"categoria": categoria, "trilha": [f"classificar -> {categoria}"]}

    def responder(estado: EstadoAtendimento) -> dict:
        """Nó 2: usa a decisão do nó anterior, que veio pelo estado."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você é um especialista da área {categoria}. Responda em uma frase."),
            ("human", "{pergunta}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke({
            "categoria": estado["categoria"],
            "pergunta": estado["pergunta"],
        })
        return {"resposta": texto, "trilha": ["responder"]}

    def auditar(estado: EstadoAtendimento) -> dict:
        """Nó 3: nem todo nó precisa de LLM. Aqui é Python puro."""
        return {"trilha": [f"auditar -> resposta com {len(estado['resposta'])} chars"]}

    grafo = StateGraph(EstadoAtendimento)
    grafo.add_node("classificar", classificar)
    grafo.add_node("responder", responder)
    grafo.add_node("auditar", auditar)

    grafo.add_edge(START, "classificar")
    grafo.add_edge("classificar", "responder")
    grafo.add_edge("responder", "auditar")
    grafo.add_edge("auditar", END)

    return grafo.compile()

def run_aula_5_video_1():
    print("--- AULA 5 / VÍDEO 01: DA CHAIN LINEAR À MÁQUINA DE ESTADOS ---")

    llm = criar_llm()
    pergunta = "Nosso faturamento caiu 8% no trimestre. O que devo olhar primeiro?"

    print("\n=== 1. LCEL: entra string, sai string ===")
    print(f"Pergunta: {pergunta}")
    print(f"Resposta: {executar_lcel(llm, pergunta)}")
    print("\nNão sobrou nada: nenhuma decisão intermediária, nenhum rastro.")

    app = criar_app(llm)

    print("\n=== 2. O desenho do grafo ===")
    print(app.get_graph().draw_mermaid())
    print("Cole isso em https://mermaid.live para ver o diagrama.")

    print("\n=== 3. StateGraph: entra estado, sai estado ===")
    estado_final = app.invoke({
        "pergunta": pergunta,
        "categoria": "",
        "resposta": "",
        "trilha": [],
    })

    print(f"Categoria decidida: {estado_final['categoria']}")
    print(f"Resposta:           {estado_final['resposta']}")
    print("\nTrilha acumulada pelo reducer:")
    for passo in estado_final["trilha"]:
        print(f"  - {passo}")

    print("\n=== 4. Vendo o estado a cada passo ===")
    for parcial in app.stream(
        {"pergunta": pergunta, "categoria": "", "resposta": "", "trilha": []},
        stream_mode="updates",
    ):
        for no, mudanca in parcial.items():
            print(f"  [{no}] devolveu: {list(mudanca.keys())}")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_1()

## video02

In [ ]:
import operator
import os
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

CAMINHO_BANCO = Path(__file__).parent / "memoria_grafo_aula5.db"

MAX_TENTATIVAS = 3

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

class EstadoRedacao(TypedDict):
    tema: str
    rascunho: str
    critica: str
    aprovado: bool
    tentativas: int
    trilha: Annotated[list[str], operator.add]

def criar_app(llm, checkpointer):
    def gerar(estado: EstadoRedacao) -> dict:
        """Escreve (ou reescreve, se já houve crítica)."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Escreva UMA frase de resumo executivo sobre o tema. "
             "Se houver crítica anterior, corrija exatamente o que ela apontou."),
            ("human", "Tema: {tema}\nCrítica anterior: {critica}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke({
            "tema": estado["tema"],
            "critica": estado["critica"] or "(nenhuma, é a primeira versão)",
        })
        tentativa = estado["tentativas"] + 1
        return {
            "rascunho": texto,
            "tentativas": tentativa,
            "trilha": [f"gerar (tentativa {tentativa})"],
        }

    def revisar(estado: EstadoRedacao) -> dict:
        """O agente critica o próprio trabalho. É a autocrítica do loop."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você revisa resumos executivos. Se o texto tiver UMA frase e citar "
             "um número, responda exatamente APROVADO. Caso contrário, responda "
             "com uma frase dizendo o que corrigir."),
            ("human", "{rascunho}"),
        ])
        parecer = (prompt | llm | StrOutputParser()).invoke(
            {"rascunho": estado["rascunho"]}
        ).strip()

        aprovado = parecer.upper().startswith("APROVADO")
        return {
            "critica": "" if aprovado else parecer,
            "aprovado": aprovado,
            "trilha": [f"revisar -> {'aprovado' if aprovado else 'reprovado'}"],
        }

    def publicar(estado: EstadoRedacao) -> dict:
        """Nó de alta criticidade: é aqui que o grafo vai pausar."""
        return {"trilha": [f"publicar -> {estado['rascunho'][:40]}..."]}

    def decidir(estado: EstadoRedacao) -> str:
        """ARESTA CONDICIONAL: calcula o próximo nó lendo o estado.

        Note o teto de tentativas. Sem ele, um revisor exigente demais deixa
        o grafo girando para sempre — e queimando tokens.
        """
        if estado["aprovado"]:
            return "publicar"
        if estado["tentativas"] >= MAX_TENTATIVAS:
            return "publicar"   # desiste de refinar e segue com o que tem
        return "gerar"          # <- a aresta que volta para trás: o ciclo

    grafo = StateGraph(EstadoRedacao)
    grafo.add_node("gerar", gerar)
    grafo.add_node("revisar", revisar)
    grafo.add_node("publicar", publicar)

    grafo.add_edge(START, "gerar")
    grafo.add_edge("gerar", "revisar")
    grafo.add_conditional_edges("revisar", decidir, ["gerar", "publicar"])
    grafo.add_edge("publicar", END)

    # interrupt_before: o grafo para ANTES de executar 'publicar' e devolve o
    # controle. Nada é publicado sem alguém olhar.
    return grafo.compile(checkpointer=checkpointer, interrupt_before=["publicar"])


def run_aula_5_video_2():
    print("--- AULA 5 / VÍDEO 02: CICLO, PERSISTÊNCIA E HUMAN-IN-THE-LOOP ---")

    llm = criar_llm()
    configuracao = {"configurable": {"thread_id": "redacao_q1"}}
    entrada = {
        "tema": "queda de 8% no faturamento do trimestre",
        "rascunho": "", "critica": "", "aprovado": False,
        "tentativas": 0, "trilha": [],
    }

    with SqliteSaver.from_conn_string(str(CAMINHO_BANCO)) as checkpointer:
        app = criar_app(llm, checkpointer)

        print("\n=== 1. O desenho do grafo (repare na aresta que volta) ===")
        print(app.get_graph().draw_mermaid())

        print("\n=== 2. Rodando até a pausa ===")
        for parcial in app.stream(entrada, config=configuracao, stream_mode="updates"):
            for no, mudanca in parcial.items():
                if no == "__interrupt__":       # o grafo bateu no interrupt_before
                    print("  [PAUSA] interrompido antes de 'publicar'")
                    continue
                print(f"  [{no}] {mudanca['trilha'][0]}")

        print("\n=== 3. O grafo PAUSOU. Estado na mão do humano ===")
        estado = app.get_state(configuracao)
        print(f"  Próximo nó a executar: {estado.next}")
        print(f"  Tentativas até aqui:   {estado.values['tentativas']}")
        print(f"  Rascunho proposto:     {estado.values['rascunho']}")

        print("\n=== 4. OVERRIDE: o humano edita o estado antes de liberar ===")
        app.update_state(
            configuracao,
            {"rascunho": estado.values["rascunho"] + " (revisado por Adriano)"},
        )
        print(f"  Rascunho após edição:  {app.get_state(configuracao).values['rascunho']}")

        print("\n=== 5. APPROVAL: invoke(None) retoma de onde parou ===")
        final = app.invoke(None, config=configuracao)
        print("  Trilha completa:")
        for passo in final["trilha"]:
            print(f"    - {passo}")

        print("\n=== 6. TIME TRAVEL: os snapshots salvos pelo checkpointer ===")
        historico = list(app.get_state_history(configuracao))
        print(f"  {len(historico)} snapshots gravados (do mais recente ao início):")
        for snap in historico[:6]:
            proximo = snap.next or ("FIM",)
            print(f"    próximo={proximo[0]:<12} tentativas={snap.values.get('tentativas')}")
        print("\n  Cada um desses tem um checkpoint_id: dá para reexecutar a partir")
        print("  de qualquer ponto passando-o no config.")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_2()

## video03

In [ ]:
import operator
import os
import time
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

ESPECIALISTAS = ("pesquisador", "analista", "redator")
MAX_PASSOS = 6  # trava contra supervisor indeciso girando para sempre

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

# ===========================================================================
# 1. Padrão Supervisor
# ===========================================================================

class EstadoEquipe(TypedDict):
    tarefa: str
    proximo: str
    passos: int
    entregas: Annotated[list[str], operator.add]
    trilha: Annotated[list[str], operator.add]

def criar_app_supervisor(llm):
    def supervisor(estado: EstadoEquipe) -> dict:
        """Decide quem trabalha agora. Não executa nada — só roteia."""
        ja_feito = [e.split(":")[0] for e in estado["entregas"]]

        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você coordena uma equipe. Especialistas disponíveis: {equipe}.\n"
             "Já entregaram: {feito}.\n"
             "Responda com UMA palavra: o próximo especialista, ou FINISH se a "
             "tarefa estiver completa. Só a palavra."),
            ("human", "Tarefa: {tarefa}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke({
            "equipe": ", ".join(ESPECIALISTAS),
            "feito": ", ".join(ja_feito) or "ninguém ainda",
            "tarefa": estado["tarefa"],
        }).strip().lower()

        # O supervisor é um LLM: valide a decisão dele contra a lista real.
        proximo = bruto if bruto in ESPECIALISTAS else "FINISH"

        return {
            "proximo": proximo,
            "passos": estado["passos"] + 1,
            "trilha": [f"supervisor -> {proximo}"],
        }

    def criar_especialista(nome: str, instrucao: str):
        def no(estado: EstadoEquipe) -> dict:
            prompt = ChatPromptTemplate.from_messages([
                ("system", instrucao + " Responda em uma frase."),
                ("human", "Tarefa: {tarefa}"),
            ])
            texto = (prompt | llm | StrOutputParser()).invoke(
                {"tarefa": estado["tarefa"]}
            )
            return {"entregas": [f"{nome}: {texto}"], "trilha": [nome]}
        return no

    def rotear(estado: EstadoEquipe) -> str:
        """ARESTA CONDICIONAL: lê a decisão do supervisor e devolve o nó."""
        if estado["passos"] >= MAX_PASSOS:
            return END
        if estado["proximo"] in ESPECIALISTAS:
            return estado["proximo"]
        return END

    grafo = StateGraph(EstadoEquipe)
    grafo.add_node("supervisor", supervisor)
    grafo.add_node("pesquisador",
                   criar_especialista("pesquisador", "Você levanta dados e fontes."))
    grafo.add_node("analista",
                   criar_especialista("analista", "Você interpreta números e aponta causas."))
    grafo.add_node("redator",
                   criar_especialista("redator", "Você escreve o texto final para a diretoria."))

    grafo.add_edge(START, "supervisor")
    grafo.add_conditional_edges("supervisor", rotear, [*ESPECIALISTAS, END])

    for nome in ESPECIALISTAS:
        grafo.add_edge(nome, "supervisor")

    return grafo.compile()

# ===========================================================================
# 2. Latência: sequencial vs paralelo
# ===========================================================================

class EstadoConsulta(TypedDict):
    tarefa: str
    fontes: Annotated[list[str], operator.add]


def _consulta_lenta(nome: str, segundos: float = 1.0):
    """Simula uma chamada de API ou LLM que demora."""
    def no(estado: EstadoConsulta) -> dict:
        time.sleep(segundos)
        return {"fontes": [nome]}
    return no

def criar_app_sequencial():
    grafo = StateGraph(EstadoConsulta)
    grafo.add_node("crm", _consulta_lenta("crm"))
    grafo.add_node("erp", _consulta_lenta("erp"))
    grafo.add_edge(START, "crm")
    grafo.add_edge("crm", "erp")      # erp só começa quando crm termina
    grafo.add_edge("erp", END)
    return grafo.compile()

def criar_app_paralelo():
    grafo = StateGraph(EstadoConsulta)
    grafo.add_node("crm", _consulta_lenta("crm"))
    grafo.add_node("erp", _consulta_lenta("erp"))
    grafo.add_edge(START, "crm")
    grafo.add_edge(START, "erp")      # duas arestas saindo do START = fan-out
    grafo.add_edge("crm", END)
    grafo.add_edge("erp", END)
    return grafo.compile()

def run_aula_5_video_3():
    print("--- AULA 5 / VÍDEO 03: MULTI-AGENTES E ENGENHARIA ---")

    llm = criar_llm()
    app = criar_app_supervisor(llm)

    print("\n=== 1. O desenho da equipe ===")
    print(app.get_graph().draw_mermaid())

    print("\n=== 2. O supervisor distribuindo o trabalho ===")
    entrada = {
        "tarefa": "Explicar a queda de 8% no faturamento do trimestre",
        "proximo": "", "passos": 0, "entregas": [], "trilha": [],
    }
    final = app.invoke(entrada)

    for passo in final["trilha"]:
        print(f"  {passo}")

    print("\n=== 3. O que cada especialista entregou ===")
    for entrega in final["entregas"]:
        print(f"  - {entrega}")

    print("\n=== 4. Latência: dois nós independentes ===")
    entrada_consulta = {"tarefa": "consolidar dados", "fontes": []}

    inicio = time.perf_counter()
    r1 = criar_app_sequencial().invoke(entrada_consulta)
    seq = time.perf_counter() - inicio

    inicio = time.perf_counter()
    r2 = criar_app_paralelo().invoke(entrada_consulta)
    par = time.perf_counter() - inicio

    print(f"  sequencial (crm -> erp): {seq:.2f}s  fontes={r1['fontes']}")
    print(f"  paralelo   (crm | erp):  {par:.2f}s  fontes={r2['fontes']}")
    print("\n  A diferença é só o desenho das arestas. Quando dois nós não")
    print("  dependem um do outro, ligue os dois ao mesmo nó anterior.")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_3()